# Ch 14: NLP w/ RNNs + Attention

NLP used to be dominated by RNNs, now transformers dominate, but SSMs, space state models, are making a comeback.

## Generating Shakespere w/ Character RNN

### Creating the Training Dataset

In [9]:
from pathlib import Path
import urllib.request

def download_shakespeare_text():
    path = Path("datasets/shakespeare/shakespeare.txt")
    if not path.is_file():
        path.parent.mkdir(parents=True, exist_ok=True)
        url = "https://homl.info/shakespeare"
        urllib.request.urlretrieve(url, path)
    return path.read_text()

shakespeare_text = download_shakespeare_text()
print(shakespeare_text[:100])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


Step 1: turn text into numbers. Usually this is done by splitting words/characters into tokens and tokenizing, assigning an ID to every possible token

To reduce size, use lowercase only:

In [10]:
vocabulary = sorted(set(shakespeare_text.lower()))

# and also make a convenient way to convert

char_to_id = {char: index for index,char in enumerate(vocabulary)}
id_to_char = {index: char for index,char in enumerate(vocabulary)}

char_to_id["k"]

23

In [11]:
# The same thing, but with torch tensors
import torch

def encode_text(text):
    return torch.tensor([char_to_id[char] for char in text.lower()])

def decode_text(ids):
    return "".join([id_to_char[id.item()] for id in ids])

encoded = encode_text("Hello World")
decoded = decode_text(encoded)
decoded

'hello world'

In [12]:
# Prepping our dataset
from torch.utils.data import DataLoader, Dataset

class CharDataset(Dataset):
    def __init__(self, text, window_length):
        self.encoded_text = encode_text(text)
        self.window_length = window_length

    def __len__(self):
        return len(self.encoded_text) - self.window_length
    
    def __getitem__(self, idx):
        if idx >= len(self):
            raise IndexError
        end = idx + self.window_length
        window = self.encoded_text[idx:end]
        target = self.encoded_text[idx+1:end+1] # Target is window shifted one right
        return window, target

In [13]:
window = 50
batch_size = 256
train_set = CharDataset(shakespeare_text[:1_000_000], window)
valid_set = CharDataset(shakespeare_text[1_000_000:1_060_000], window)
test_set = CharDataset(shakespeare_text[1_060_000:], window)
train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
valid_loader = DataLoader(valid_set, batch_size=batch_size)
test_loader = DataLoader(test_set, batch_size=batch_size)

Neural nets come with the underlying assumption that closer number = better. In this case, that doesn't work, since "a" is not necessarily closer to "b" than "i" when trying to guess the next letter of a word. 

You could OHE -> but that explodes data size for big vocabularies

Instead: embed

### Embeddings

An embedding is a dense representation of higher-dim data, usually a cat feature. High number of possible categories causes OHE to produce huge vector, while embedding will produce a smaller vector. 

Usually make embedding size = root n_categories

For DL, embeddings can be initialized randomly, then trained via gradient descent.

Hopefully after training, similar categories are closer together, and unrelated concepts are far apart. This is representation learning

Embeddings can be reused for other tasks -> usually better to get someone elses trained embedding than using your own

Embeddings can be organized along useful axes. For ex, the embeddings of King - Man + Woman roughly equal the embedding of Queen. 

Pytorch has nn.Embedding, a wrapped of a matrix with one row per category, and one col per embedding dimension. To get the embedding, it just looks up whatever row you want.

In [14]:
import torch.nn as nn

torch.manual_seed(22)
embed = nn.Embedding(5,3) 
embed(torch.tensor([[3,2],[0,2]]))
# so cat 3 is embedded as [1.3164, -0.8576, -0.3336]

tensor([[[ 1.3164, -0.8576, -0.3336],
         [ 0.7240,  0.2416, -1.5601]],

        [[ 1.0302, -0.5073, -0.1017],
         [ 0.7240,  0.2416, -1.5601]]], grad_fn=<EmbeddingBackward0>)

An embedding layer is the same as OHE followed by a linear layer (w/o bias): You are just multiplying the OHE vector by the embedding matrix, but you dont have to do a bunch of multiplications by 0

### Building and Training Char-RNN

Large dataset, need some complexity, so probably need GRU instead of a simple RNN

In [15]:
class ShakespeareModel(nn.Module):
    def __init__(self, vocab_size, n_layers=2, embed_dim=10, hidden_dim=128, dropout=0.1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim) # encodes character IDs. you can tune embedding dim
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers, batch_first=True, dropout=dropout) #need to set batch first, otherwise layer assumes batch dim after time dim
        self.output = nn.Linear(hidden_dim, vocab_size) # needs one output per distinct char

    def forward(self, X):
        embeddings = self.embed(X)
        outputs, _states = self.gru(embeddings)
        return self.output(outputs).permute(0,2,1) # cross entropy and accuracy expect batch size, class, window length
                                                 # but linear will output batch size, window length, vocab size since GRU outputs batch, window, hidden
    
device = "cuda"
model = ShakespeareModel(len(vocabulary)).to(device)

In [16]:
# !pip install torchmetrics
import torchmetrics
loss = nn.CrossEntropyLoss()
metric = torchmetrics.Accuracy(task="multiclass", num_classes=len(vocabulary)).to(device)
optim = torch.optim.AdamW(model.parameters())

def train(model, dataloader, loss_fn, metric, optim, n_epochs):
    model.train()
    for epoch in range(n_epochs):
        epoch_stats = []
        for X_train, y_train in dataloader:
            X_train = X_train.to(device)
            y_train = y_train.to(device)
            y_hat = model(X_train)
            loss = loss_fn(y_hat,y_train)
            optim.zero_grad()
            loss.backward()
            optim.step()
            epoch_stats.append(metric(y_hat,y_train))
        print(epoch, sum(epoch_stats)/len(epoch_stats))

train(model, train_loader, loss, metric, optim, 1)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 27.6 MB/s eta 0:00:0000:01
0 tensor(0.5175, device='cuda:0')


In [17]:
# got ~56% with 7 epochs
model.eval()  
text = "That is the mod"
encoded_text = encode_text(text).unsqueeze(dim=0).to(device)
with torch.no_grad():
    Y_logits = model(encoded_text)
    predicted_char_id = Y_logits[0, :, -1].argmax().item()
    predicted_char = id_to_char[predicted_char_id]

predicted_char

'e'

### Generating Text

Greedy decoding: model predicts next character, then you feed the input+predicted char back into the model. In practice, leads to high repetition

Instead, pick the next character randomly: if the model predicts "e" follows "To be or not to b" with p=.9 and k with p=.1, we randomly select e or k.

In [18]:
probs = torch.tensor([[.5,.4,.1]])
torch.multinomial(probs, replacement=True, num_samples=10)
#will pick 0 or 1 most of the time since they have the highest weights

tensor([[2, 2, 1, 0, 1, 2, 1, 0, 0, 1]])

Add control by dividing logits by a new parameter, temperature. High temperature = more random, and low temp = less random. 

In [19]:
import torch.nn.functional as F

def next_char(model, text, temperature=1):
    encoded_txt = encode_text(text).unsqueeze(dim=0).to(device) #unsqueeze adds a batch dim
    with torch.no_grad():
        Y_logits = model(encoded_txt)
        Y_probas = F.softmax(Y_logits[0,:,-1] / temperature, dim=-1) # 0 drops batch dim, -1 grabs logits
                                                                    # and softmax converts to probas
        pred_id = torch.multinomial(Y_probas, num_samples=1).item()
    return id_to_char[int(pred_id)]

def extend_text(model, text, n_chars=80, temperature=1):
    for _ in range(n_chars):
        text+=next_char(model, text, temperature)
    return text

print(extend_text(model, "To be or not to b", temperature=.9))

To be or not to be is
a cousin, hear, which is once day and marshal.

prince:
o, you'll commis do


You could also try only sampling the top few predicted probability characters (top k), or use beam search, or adding more GRU layers, or increasing neurons, or training for longer.

Notably, very large RNNs can learn something similar to sentiment analysis, like for the sentence "That movie was great, I really ", it predicts L (liked, loved) is way more likely than H (hated). 

## Sentiment Analysis w/ HuggingFace Libraries

In [20]:
from datasets import load_dataset

imdb_dataset = load_dataset("imdb")
split = imdb_dataset["train"].train_test_split(train_size=0.8, seed=42)
imdb_train_set, imdb_valid_set = split["train"], split["test"]
imdb_test_set = imdb_dataset["test"]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

plain_text/train-00000-of-00001.parquet:   0%|          | 0.00/21.0M [00:00<?, ?B/s]

plain_text/test-00000-of-00001.parquet:   0%|          | 0.00/20.5M [00:00<?, ?B/s]

plain_text/unsupervised-00000-of-00001.p(…):   0%|          | 0.00/42.0M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [21]:
imdb_train_set[0]["text"]

'Stage adaptations often have a major fault. They often come out looking like a film camera was simply placed on the stage (Such as "Night Mother"). Sidney Lumet\'s direction keeps the film alive, which is especially difficult since the picture offered him no real challenge. Still, it\'s nice to look at for what it is. The chemistry between Michael Caine and Christopher Reeve is quite brilliant. The dynamics of their relationship are surprising. Caine is fantastic as always, and Reeve gets one of his few chances to really act.<br /><br />I confess that I\'ve never seen Ira Levin\'s play, but I hear that Jay Presson Allen\'s adaptation is faithful. The script is incredibly convoluted, and keeps you guessing. "Deathtrap" is an enormously entertaining film, and is recommended for nearly all fans of stage and screen.<br /><br />7.4 out of 10'

This example is somewhat difficult, it starts off by talking about difficulties and "major fault". Still, it's a positive review.

We need to tokenize instead of going char-by-char

### Tokenization w/ HF Tokenizers Library

Split words into tokens: smartest becomes "smart" and "est". How? One way is byte pair encoding, BPE. Split your training set into individual characters, then at each iteration find the most frequent pair of adjacent tokens and add it to the vocab.

In [22]:
import tokenizers

bpe_model = tokenizers.models.BPE(unk_token="<unk>") #if token is unknown, replace it w/ unk
bpe_tokenizer = tokenizers.Tokenizer(bpe_model)
bpe_tokenizer.pre_tokenizer = tokenizers.pre_tokenizers.Whitespace() #splits text at spaces, and separates letters/nonletters
special_tokens = ["<pad>", "<unk>"]
bpe_trainer = tokenizers.trainers.BpeTrainer(vocab_size=1000, special_tokens=special_tokens)
train_reviews = [review["text"].lower() for review in imdb_train_set]
bpe_tokenizer.train_from_iterator(train_reviews, bpe_trainer)

In [23]:
review_1 = "that movie was sooooooo sick%"
bpe_tokenizer.encode(review_1).tokens
# tokenizer recognizes "that" and "movie", but not "sick" or "sooooooo"

['that', 'movie', 'was', 'so', 'o', 'o', 'o', 'o', 'o', 'o', 's', 'ick', '%']

In [24]:
token_ids = bpe_tokenizer.encode(review_1).ids
bpe_tokenizer.decode(token_ids)
# get_vocab() returns a token-to-id dictionary

'that movie was so o o o o o o s ick %'

In [25]:
bpe_tokenizer.encode(review_1).offsets
# helpful for debugging where your tokenizer is messing up

[(0, 4),
 (5, 10),
 (11, 14),
 (15, 17),
 (17, 18),
 (18, 19),
 (19, 20),
 (20, 21),
 (21, 22),
 (22, 23),
 (24, 25),
 (25, 28),
 (28, 29)]

In [26]:
bpe_tokenizer.enable_padding(pad_id=0, pad_token="<pad>")
bpe_tokenizer.enable_truncation(max_length=500)
# to create a tensor, we need to have consistent dims, which means padding+truncation needed

In [27]:
encodings = bpe_tokenizer.encode_batch(train_reviews[:3])
bpe_batch_ids = torch.tensor([encoding.ids for encoding in encodings])

Every Encoding object (which encode_batch returns) has an attention_mask attribute, this can be used to tell your model what to ignore easily by multiplying

Or you could get the sequence lengths themselves

In [28]:
attn_mask = torch.tensor([encoding.attention_mask for encoding in encodings])
lengths = attn_mask.sum(dim=-1)

Spaces don't really work with our tokenizer, it ends up just putting spaces between every token because Whitespace gets rid of the spaces

Instead, use ByteLevel pre-tokenizer, which replaces the spaces with their own special character

ByteLevel works at the byte level, not character level, using UTF-8 encoding. There are only 256 possible bytes, so BBPE works well for long tokens not found in the vocabulary

WordPiece, a BPE variant, works by adding the pair of adjacent tokens with the highest score. The score boosts frequent pairs, but normalizes by how many times the individual tokens show up. So "the" is a very common word, but "the ----" probably doesn't need its own token. WordPiece favors more useful/meaningful tokens, and shortens the encoding sequence

Wordpiece Score Formula (for tokens A, B): $\frac{frequency(AB)}{freq(A)\cdot freq(B)} \cdot len(vocab)$

To train w/ Tokenizers library, use same code as above, but replace "BPE" model w/ WordPiece

Another tokenizer algo: UnigramLM. Starts with a vocabulary of every possible word, subword, and character in your text corpus, and then removes the least useful ones until you get the size you want. Assumes that the corpus was sampled randomly from your vocabulary, each token independent from others, so P(AB)=P(A)P(B). It can then estimate P(sampling this corpus), and remove tokens that do not reduce P(sampling this corpus) too much. UnigramLM is good for languages that don't use spaces

^ same paper into'd Subword Regularization, which improves generalization by intro'ing randomness in tokenization during NLP model training. Assume your vocabulary has: "them", "the", "m" tokens. Sometimes you will tokenize "them" as "them", sometimes "the"+"m". If a language carries a lot of info through grammatical modifications/affixes, this method can help. 

| Feature | BBPE | WordPiece | Unigram LM |
|---------|------|-----------|------------|
| **How** | Merge most frequent pairs | Merge pairs that maximize data likelihood | Remove least likely tokens |
| **Pros** | Fast, simple, great for multilingual | Good balance of efficiency and token quality | Most meaningful, shortest sequences |
| **Cons** | Can produce awkward splits | Less robust than BBPE for multilingual | Slower to train and tokenize |
| **Used By** | GPT, Llama, RoBERTa, BLOOM | BERT, DistilBERT, ELECTRA | T5, ALBERT, mBART |

Generally: best to use a pretrained tokenizer

If you have a lot of domain specific info, or a language without a lot of resources available on it, you may want to consider training your own tokenizer.

### Reusing Pretrained Tokenizers

HF transformers library can get you tokenizers as well as transformer models, cnns, various neural nets

In [36]:
import transformers

gpt2_tokenizer = transformers.AutoTokenizer.from_pretrained("gpt2")
gpt2_encoding = gpt2_tokenizer(train_reviews[:3], truncation=True, max_length=500)
gpt2_encoding["input_ids"][0][:5] # get first 5 tokens of first review

[14247, 35030, 1690, 423, 257]

In [44]:
gpt2_tokenizer.decode(gpt2_encoding["input_ids"][0][:10])

'stage adaptations often have a major fault. they often'

In [47]:
gpt2_tokenizer("malformations")

{'input_ids': [7617, 687, 602], 'attention_mask': [1, 1, 1]}

If you want a WordPiece tokenizer, you need to get a tokenizer from a model pretrained w/ wordpiece, like BERT

This tokenizer has a padding token and lets us get pytorch tensors

In [ ]:
bert_tokenizer = transformers.AutoTokenizer.from_pretrained("bert-base-uncased")
bert_encoding = bert_tokenizer(train_reviews[:3], padding=True,
                               truncation=True, max_length=500,
                               return_tensors="pt")

# bert_encoding["input_ids"] # now there is padding, and attn masks in ["attention_mask"]

### Building and Training a Sentiment Analysis Model

To tokenize your text data, you can pass a function to the DataLoader w/ the collate_fn arg. Your function needs to take in a batch, tokenize, truncate and pad if needed, and return a BatchEncoding object (that holds token ids, attn masks, labels)

In [52]:
def collate_fn(batch, tokenizer=bert_tokenizer):
    reviews = [review["text"] for review in batch]
    labels = [[review["label"]] for review in batch]
    encodings = tokenizer(reviews, padding=True, truncation=True, max_length=200, return_tensors="pt") #pytorch
    labels = torch.tensor(labels, dtype=torch.float32)
    return encodings, labels

batch_size = 256
train_loader = DataLoader(imdb_train_set, batch_size=batch_size, collate_fn=collate_fn, shuffle=True)
valid_loader = DataLoader(imdb_valid_set, batch_size=batch_size, collate_fn=collate_fn)
test_loader = DataLoader(imdb_test_set, batch_size=batch_size, collate_fn=collate_fn)

In [53]:
class SentimentAnalysisModel(nn.Module):
    def __init__(self, vocab_size, n_layers=2, embed_dim=128, hidden_dim=64, pad_id=0, dropout=0.2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_id) #need to tell embedding what the padding idx is so it knows not to train it
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=n_layers, batch_first=True, dropout=dropout)
        self.output = nn.Linear(hidden_dim, 1) #binary classification

    def forward(self, encodings): # takes in a batchencoding object
        embeddings = self.embed(encodings["input_ids"])
        _outputs, hidden_states = self.gru(embeddings)
        return self.output(hidden_states[-1]) # this is sequence-to-vector, so only output top GRU layer

In [57]:
# Let's train!

# vocabulary_len = bert_tokenizer.vocab_size
# model = SentimentAnalysisModel(vocab_size=vocabulary_len).to(device)
# loss = nn.BCEWithLogitsLoss()
# metric = torchmetrics.Accuracy(task="binary").to(device)
# optim = torch.optim.AdamW(model.parameters())

train(model, train_loader, loss, metric, optim, 2)

0 tensor(0.5648, device='cuda:0')
1 tensor(0.6319, device='cuda:0')


In [ ]:
model.eval()  
text = "That is the mod"
bert_tokenizer(text)
# encoded_text = encode_text(text).unsqueeze(dim=0).to(device)
# with torch.no_grad():
#     Y_logits = model(encoded_text)
#     predicted_char_id = Y_logits[0, :, -1].argmax().item()
#     predicted_char = id_to_char[predicted_char_id]

# predicted_char